# 02 Carry Research — Macro Metals System> **Strategy:** Carry / Term-Structure (Memory File §3.2)> **Scope:** In-sample development (2015–2022), daily & weekly rebalance> **Sub-modules:** FX Carry (G10) · Metals Calendar Spreads (GC/SI) · SOFR Curve>> Self-contained BQuant notebook — executable top-to-bottom.

In [ ]:
import warningswarnings.filterwarnings("ignore")import numpy as npimport pandas as pdimport plotly.express as pximport plotly.graph_objects as gofrom plotly.subplots import make_subplotsimport yamlfrom pathlib import Pathfrom datetime import datetime# Bloomberg BQLimport bqlbq = bql.Service()print(f"Session started : {datetime.now():%Y-%m-%d %H:%M}")print(f"BQL service     : {type(bq).__name__}")print(f"NumPy {np.__version__}  |  pandas {pd.__version__}")

## Config & ParametersLoad `parameters.yaml` and `tickers.yaml` from `config/`.Three carry sub-modules pull from `strategies.carry`:| Sub-module | Config key | Rebalance ||------------|------------|-----------|| FX Carry | `carry.fx` | Weekly || Metals Calendar Spreads | `carry.metals.calendar_spreads` | Daily || SOFR Curve | `carry.sofr_curve` | Daily |

In [ ]:
CONFIG_DIR = Path("../config")with open(CONFIG_DIR / "parameters.yaml") as f:    params = yaml.safe_load(f)with open(CONFIG_DIR / "tickers.yaml") as f:    tickers = yaml.safe_load(f)gcfg      = params["global"]carry_cfg = params["strategies"]["carry"]targets   = params["performance_targets"]fx_cfg    = carry_cfg["fx"]met_cfg   = carry_cfg["metals"]["calendar_spreads"]sofr_cfg  = carry_cfg["sofr_curve"]IS_START = gcfg["in_sample_start"]IS_END   = gcfg["in_sample_end"]print("Carry parameters:")print(f"\n  FX Carry:")print(f"    Pairs          : {fx_cfg['pairs']}")print(f"    Horizon        : {fx_cfg['carry_horizon_days']}d")print(f"    Rank top/bot   : {fx_cfg['ranking_top_pct']:.0%} / {fx_cfg['ranking_bottom_pct']:.0%}")print(f"    Min carry      : {fx_cfg['min_carry_bp']} bp")print(f"    Trend filter   : {fx_cfg['trend_filter_lookback_days']}d")print(f"    Crisis VIX     : {fx_cfg['crisis_vix_threshold']}")print(f"    Rebalance      : {fx_cfg['rebalance']}")print(f"\n  Metals Calendar Spreads:")print(f"    Instruments    : {met_cfg['instruments']}")print(f"    Z-score window : {met_cfg['zscore_lookback_days']}d")print(f"    Entry/exit z   : ±{met_cfg['entry_zscore']} / ±{met_cfg['exit_zscore']}")print(f"    Carry mom.     : {met_cfg['carry_momentum_lookback_days']}d")print(f"    Rebalance      : {met_cfg['rebalance']}")print(f"\n  SOFR Curve:")print(f"    Instruments    : {sofr_cfg['instruments']}")print(f"    Tenors         : {sofr_cfg['tenors_months']}M")print(f"    Slope thresh.  : {sofr_cfg['steepness_threshold_bp']} bp")print(f"    Stability win. : {sofr_cfg['policy_stability_window_days']}d")print(f"    Rebalance      : {sofr_cfg['rebalance']}")print(f"\n  IS period: {IS_START} to {IS_END}")

## Data Pipeline (BQL)Enhanced `BQuantDataLoader` with:- `get_history()` — single ticker daily PX_LAST- `get_curve_data()` — multi-ticker panel for term structure- `get_fx_carry_pair()` — spot + 3M forward for carry calculation

In [ ]:
class BQuantDataLoader:    """Fetch historical prices via Bloomberg BQL.    Enhanced for carry strategies with curve and FX carry helpers.    """    def __init__(self, ticker_map: dict) -> None:        self._tickers = ticker_map        self._bq = bql.Service()    def resolve(self, logical_name: str) -> str:        """Logical name -> Bloomberg ticker string."""        for group in self._tickers.values():            if isinstance(group, dict) and logical_name in group:                return group[logical_name]        raise KeyError(f"{logical_name} not in tickers.yaml")    def get_history(        self,        logical_name: str,        start: str,        end: str,        field: str = "PX_LAST",    ) -> pd.Series:        """Fetch daily price series for one instrument."""        bbg = self.resolve(logical_name)        request = bql.Request(            bbg,            {field: self._bq.data.px_last(                dates=self._bq.func.range(start, end)            )},        )        try:            response = self._bq.execute(request)            df = response[0].df()            if df.empty:                print(f"  ! {logical_name}: empty response")                return pd.Series(dtype=float, name=logical_name)            if isinstance(df.index, pd.MultiIndex):                series = df[field].droplevel("ID")            else:                series = df[field]            series = series.sort_index().astype(float)            series.name = logical_name            series.index.name = "date"            return series        except Exception as exc:            print(f"  ! {logical_name} ({bbg}): {exc}")            return pd.Series(dtype=float, name=logical_name)    def get_curve_data(        self,        logical_names: list[str],        start: str,        end: str,    ) -> pd.DataFrame:        """Fetch a panel of prices for multiple tickers (term structure).        Returns:            DataFrame indexed by date, columns = logical names.        """        panel = {}        for name in logical_names:            s = self.get_history(name, start, end)            if len(s) > 0:                panel[name] = s        return pd.DataFrame(panel).sort_index().ffill()    def get_fx_carry_pair(        self,        spot_name: str,        fwd_name: str,        start: str,        end: str,    ) -> pd.DataFrame:        """Fetch spot + 3M forward outright for one FX pair.        Returns:            DataFrame with columns: spot, forward, fwd_points.        """        spot = self.get_history(spot_name, start, end)        fwd  = self.get_history(fwd_name, start, end)        df = pd.DataFrame({"spot": spot, "forward": fwd}).dropna()        df["fwd_points"] = df["forward"] - df["spot"]        return df# --- Fetch data ---loader = BQuantDataLoader(tickers)# FX spot + forwardsFX_PAIRS = {    "eurusd": ("eurusd_spot", "eurusd_3m_fwd"),    "usdjpy": ("usdjpy_spot", "usdjpy_3m_fwd"),    "gbpusd": ("gbpusd_spot", "gbpusd_3m_fwd"),    "audusd": ("audusd_spot", "audusd_3m_fwd"),    "usdcnh": ("usdcnh_spot", "usdcnh_3m_fwd"),}# Map config pair names to our FX_PAIRS keysFX_PAIR_MAP = {    "eurusd_spot": "eurusd",    "usdjpy_spot": "usdjpy",    "gbpusd_spot": "gbpusd",    "audusd_spot": "audusd",    "usdchf_spot": None,  # no 3M forward in tickers — skip    "usdcnh_spot": "usdcnh",}print("=== FX Carry Data ===")fx_data = {}for pair_cfg in fx_cfg["pairs"]:    key = FX_PAIR_MAP.get(pair_cfg)    if key is None:        print(f"  {pair_cfg:18s} SKIP (no forward ticker)")        continue    spot_name, fwd_name = FX_PAIRS[key]    print(f"  {key:18s}", end=" ")    try:        df = loader.get_fx_carry_pair(spot_name, fwd_name, IS_START, IS_END)        fx_data[key] = df        print(f"OK  {len(df):>5d} obs  [{df.index[0]:%Y-%m-%d} -> {df.index[-1]:%Y-%m-%d}]")    except Exception as e:        print(f"FAIL: {e}")# Metals futures (front/second/third)print("\n=== Metals Calendar Spread Data ===")metals_tickers = met_cfg["instruments"]metals_prices = loader.get_curve_data(metals_tickers, IS_START, IS_END)print(f"  Panel: {metals_prices.shape[0]} days x {metals_prices.shape[1]} contracts")# SOFR futuresprint("\n=== SOFR Curve Data ===")sofr_tickers = sofr_cfg["instruments"]sofr_prices = loader.get_curve_data(sofr_tickers, IS_START, IS_END)print(f"  Panel: {sofr_prices.shape[0]} days x {sofr_prices.shape[1]} contracts")# VIX for crisis filterprint("\n=== Risk Indicators ===")vix = loader.get_history("vix", IS_START, IS_END)print(f"  VIX: {len(vix)} obs")# DXY for trend filterdxy = loader.get_history("dxy_index", IS_START, IS_END)print(f"  DXY: {len(dxy)} obs")

## Carry StrategyThree sub-modules, each generating daily signals in [-1, +1]:### A. FX Carry1. Compute annualised implied carry from 3M forward points2. Cross-sectional rank — long top 30%, short bottom 30%3. Trend filter: only take positions aligned with 63-day price momentum4. Crisis filter: flatten when VIX > 30### B. Metals Calendar Spreads1. Compute spread = front − deferred contract price2. Z-score the spread vs 120-day rolling mean/std3. Enter at ±1.5σ, exit at ±0.5σ4. Directional filter: 20-day carry momentum confirmation### C. SOFR Curve1. Compute slope between adjacent quarterly futures2. Z-score vs 60-day window3. Steepener when slope z > threshold; flattener when z < −threshold

In [ ]:
class CarryStrategy:    """Carry / term-structure strategy from memory file §3.2.    Three sub-modules: FX carry, metals calendar spreads, SOFR curve.    """    def __init__(self, params: dict) -> None:        self.carry_cfg = params["strategies"]["carry"]        self.fx_cfg    = self.carry_cfg["fx"]        self.met_cfg   = self.carry_cfg["metals"]["calendar_spreads"]        self.sofr_cfg  = self.carry_cfg["sofr_curve"]    # ── A. FX Carry ────────────────────────────────────────────────    def fx_carry_signals(        self,        fx_data: dict[str, pd.DataFrame],        vix: pd.Series,    ) -> pd.DataFrame:        """Generate cross-sectional FX carry signals.        Args:            fx_data: Dict mapping pair name -> DataFrame (spot, forward, fwd_points).            vix:     Daily VIX series for crisis filter.        Returns:            DataFrame: daily signals per pair in [-1, +1].        """        horizon_days = self.fx_cfg["carry_horizon_days"]        top_pct      = self.fx_cfg["ranking_top_pct"]        bot_pct      = self.fx_cfg["ranking_bottom_pct"]        min_carry_bp = self.fx_cfg["min_carry_bp"]        trend_lb     = self.fx_cfg["trend_filter_lookback_days"]        crisis_vix   = self.fx_cfg["crisis_vix_threshold"]        use_crisis   = self.fx_cfg.get("crisis_filter", True)        # 1. Annualised implied carry: (spot - forward) / spot × (365 / horizon)        carry_df = pd.DataFrame()        spot_df  = pd.DataFrame()        for pair, df in fx_data.items():            # Carry = (spot - fwd) / spot × annualise            implied_carry = (-df["fwd_points"] / df["spot"]) * (365 / horizon_days)            carry_df[pair] = implied_carry            spot_df[pair]  = df["spot"]        # Build common index        idx = carry_df.index.intersection(vix.index)        carry_df = carry_df.reindex(idx)        spot_df  = spot_df.reindex(idx)        vix_aligned = vix.reindex(idx).ffill()        # 2. Cross-sectional ranking        ranks = carry_df.rank(axis=1, pct=True)        n_pairs = carry_df.shape[1]        signals = pd.DataFrame(0.0, index=idx, columns=carry_df.columns)        for col in carry_df.columns:            # Long top, short bottom            sig = pd.Series(0.0, index=idx)            sig[ranks[col] >= (1 - top_pct)] = 1.0            sig[ranks[col] <= bot_pct]        = -1.0            # Min carry filter            sig[carry_df[col].abs() * 10_000 < min_carry_bp] = 0.0            # 3. Trend filter: 63d momentum alignment            spot_mom = spot_df[col].pct_change(trend_lb)            # Only take carry if momentum agrees with signal direction            sig[(sig > 0) & (spot_mom < 0)] = 0.0            sig[(sig < 0) & (spot_mom > 0)] = 0.0            signals[col] = sig        # 4. Crisis filter: flatten all if VIX > threshold        if use_crisis:            crisis_mask = vix_aligned > crisis_vix            signals.loc[crisis_mask] = 0.0        # Weekly rebalance: hold positions for 5 days        # Resample to weekly, forward-fill within week        weekly_idx = signals.resample("W-FRI").last().index        signals_weekly = signals.reindex(weekly_idx).ffill()        signals = signals_weekly.reindex(idx, method="ffill").fillna(0.0)        return signals    # ── B. Metals Calendar Spreads ─────────────────────────────────    def metals_spread_signals(        self,        prices: pd.DataFrame,    ) -> tuple[pd.DataFrame, pd.DataFrame]:        """Generate calendar spread z-score signals for metals.        Args:            prices: DataFrame with front/second/third contracts as columns.        Returns:            Tuple of (signals DataFrame, z-scores DataFrame) for analysis.        """        zscore_lb = self.met_cfg["zscore_lookback_days"]        entry_z   = self.met_cfg["entry_zscore"]        exit_z    = self.met_cfg["exit_zscore"]        mom_lb    = self.met_cfg["carry_momentum_lookback_days"]        # Build spread pairs: front - second, second - third        spread_pairs = {            "GC_1v2": ("gc_fut_front", "gc_fut_second"),            "GC_2v3": ("gc_fut_second", "gc_fut_third"),            "SI_1v2": ("si_fut_front", "si_fut_second"),            "SI_2v3": ("si_fut_second", "si_fut_third"),        }        signals  = pd.DataFrame(index=prices.index)        zscores  = pd.DataFrame(index=prices.index)        for label, (front, back) in spread_pairs.items():            if front not in prices.columns or back not in prices.columns:                continue            spread = prices[front] - prices[back]            # Rolling z-score            mu  = spread.rolling(zscore_lb, min_periods=zscore_lb // 2).mean()            sig = spread.rolling(zscore_lb, min_periods=zscore_lb // 2).std()            sig = sig.replace(0, np.nan)            z   = (spread - mu) / sig            zscores[label] = z            # Signal generation with hysteresis            signal = pd.Series(0.0, index=prices.index)            position = 0.0            for i in range(len(z)):                if pd.isna(z.iloc[i]):                    signal.iloc[i] = 0.0                    continue                zv = z.iloc[i]                if position == 0.0:                    if zv > entry_z:                        position = -1.0    # Spread rich → sell spread                    elif zv < -entry_z:                        position = 1.0     # Spread cheap → buy spread                elif position > 0 and zv > -exit_z:                    position = 0.0         # Exit long                elif position < 0 and zv < exit_z:                    position = 0.0         # Exit short                signal.iloc[i] = position            # Carry momentum confirmation (20d)            carry_mom = spread.pct_change(mom_lb)            # Dampen signal if momentum contradicts            signal[(signal > 0) & (carry_mom < 0)] *= 0.5            signal[(signal < 0) & (carry_mom > 0)] *= 0.5            signals[label] = signal        return signals, zscores    # ── C. SOFR Curve ──────────────────────────────────────────────    def sofr_curve_signals(        self,        prices: pd.DataFrame,    ) -> tuple[pd.DataFrame, pd.DataFrame]:        """Generate SOFR curve steepener/flattener signals.        SOFR futures are quoted as 100 - rate, so higher price = lower rate.        Slope = front price - back price (positive = inverted / front rate lower).        Args:            prices: DataFrame with SOFR quarterly futures columns.        Returns:            Tuple of (signals DataFrame, slopes DataFrame).        """        zscore_lb    = self.sofr_cfg["zscore_lookback_days"]        threshold_bp = self.sofr_cfg["steepness_threshold_bp"]        stability    = self.sofr_cfg["policy_stability_window_days"]        contracts = [c for c in self.sofr_cfg["instruments"] if c in prices.columns]        if len(contracts) < 2:            return pd.DataFrame(), pd.DataFrame()        slopes  = pd.DataFrame(index=prices.index)        signals = pd.DataFrame(index=prices.index)        # Adjacent contract slopes        for i in range(len(contracts) - 1):            front = contracts[i]            back  = contracts[i + 1]            label = f"SOFR_{i+1}v{i+2}"            # Slope in implied-rate space: (100 - front) - (100 - back) = back - front            slope = prices[back] - prices[front]  # positive = steeper            slopes[label] = slope            # Rolling z-score            mu  = slope.rolling(zscore_lb, min_periods=zscore_lb // 2).mean()            sig = slope.rolling(zscore_lb, min_periods=zscore_lb // 2).std()            sig = sig.replace(0, np.nan)            z   = (slope - mu) / sig            # Signal: steepener when z > 0 (slope above mean), flattener when z < 0            signal = np.tanh(z)            # Policy stability filter: suppress signal if slope has been            # very volatile over the stability window            slope_vol = slope.rolling(stability).std()            slope_vol_z = (slope_vol - slope_vol.rolling(zscore_lb).mean()) / slope_vol.rolling(zscore_lb).std().replace(0, np.nan)            signal[slope_vol_z.abs() > 2.0] *= 0.5  # dampen during unstable periods            signals[label] = signal        # Butterfly: front - 2*mid + back (if 3+ contracts)        if len(contracts) >= 3:            fly = prices[contracts[0]] - 2 * prices[contracts[1]] + prices[contracts[2]]            slopes["SOFR_fly"] = fly            mu  = fly.rolling(zscore_lb, min_periods=zscore_lb // 2).mean()            sig = fly.rolling(zscore_lb, min_periods=zscore_lb // 2).std().replace(0, np.nan)            z = (fly - mu) / sig            signals["SOFR_fly"] = np.tanh(z)        return signals, slopes# Build strategycarry = CarryStrategy(params)print("CarryStrategy initialised with 3 sub-modules.")

## Generate Signals (IS Period)Run all three carry sub-modules and inspect signal distributions.

In [ ]:
# A. FX Carryprint("=== FX Carry Signals ===")fx_signals = carry.fx_carry_signals(fx_data, vix)print(f"  Shape: {fx_signals.shape}")print(f"  Active pairs: {(fx_signals != 0).any().sum()}")print(f"  Crisis days (VIX>{fx_cfg['crisis_vix_threshold']}): {(vix > fx_cfg['crisis_vix_threshold']).sum()}")print()# B. Metals Calendar Spreadsprint("=== Metals Calendar Spread Signals ===")metals_signals, metals_zscores = carry.metals_spread_signals(metals_prices)print(f"  Shape: {metals_signals.shape}")for col in metals_signals.columns:    n_trades = (metals_signals[col].diff().abs() > 0).sum()    print(f"  {col}: {n_trades} signal changes")print()# C. SOFR Curveprint("=== SOFR Curve Signals ===")sofr_signals, sofr_slopes = carry.sofr_curve_signals(sofr_prices)print(f"  Shape: {sofr_signals.shape}")print()# Summary statisticsprint("Signal statistics:")all_sigs = {    "FX Carry": fx_signals,    "Metals Spreads": metals_signals,    "SOFR Curve": sofr_signals,}for name, df in all_sigs.items():    if df.empty:        print(f"  {name}: no signals")        continue    avg_abs = df.abs().mean().mean()    pct_flat = (df == 0).mean().mean()    print(f"  {name}: avg |signal| = {avg_abs:.3f}, flat = {pct_flat:.0%}")

## Backtest EngineSame vectorised engine as momentum notebook:- Vol-target position sizing (10% annual, 30d EWMA, 2x cap)- 2 bp transaction costs per side- One-day signal lag

In [ ]:
def backtest_single_asset(    prices: pd.Series,    signals: pd.Series,    vol_target: float = 0.10,    vol_lookback: int = 30,    vol_cap: float = 2.0,    tc_bp: float = 2.0,    capital: float = 1_000_000.0,) -> pd.DataFrame:    """Vectorised single-asset backtest."""    ret = prices.pct_change().fillna(0.0)    ann_vol = ret.ewm(halflife=vol_lookback).std() * np.sqrt(252)    ann_vol = ann_vol.replace(0, np.nan)    vol_scale = (vol_target / ann_vol).clip(upper=vol_cap).fillna(1.0)    position = (signals.shift(1).fillna(0.0) * vol_scale).clip(-vol_cap, vol_cap)    ret_gross = position * ret    turnover  = position.diff().abs().fillna(0.0)    cost      = turnover * (tc_bp / 10_000)    ret_net   = ret_gross - cost    equity    = capital * (1 + ret_net).cumprod()    return pd.DataFrame({        "position": position, "price": prices,        "ret_gross": ret_gross, "ret_net": ret_net,        "equity": equity, "turnover": turnover,    })def backtest_spread(    spread: pd.Series,    signals: pd.Series,    vol_target: float = 0.10,    vol_lookback: int = 30,    vol_cap: float = 2.0,    tc_bp: float = 2.0,    capital: float = 1_000_000.0,) -> pd.DataFrame:    """Backtest a spread trade (PnL from spread changes, not returns).    Uses dollar-vol targeting instead of percentage returns.    """    spread = spread.dropna()    signals = signals.reindex(spread.index).fillna(0.0)    # Dollar vol targeting    spread_std = spread.diff().ewm(halflife=vol_lookback).std()    spread_std = spread_std.replace(0, np.nan)    target_dollar_vol = capital * vol_target / np.sqrt(252)    vol_scale = (target_dollar_vol / spread_std).clip(upper=vol_cap * capital).fillna(0.0)    position = signals.shift(1).fillna(0.0) * vol_scale / capital    position = position.clip(-vol_cap, vol_cap)    # PnL from spread changes    spread_chg = spread.diff().fillna(0.0)    ret_gross = position * spread_chg / spread.abs().clip(lower=1.0)    turnover = position.diff().abs().fillna(0.0)    cost     = turnover * (tc_bp / 10_000)    ret_net  = ret_gross - cost    equity   = capital * (1 + ret_net).cumprod()    return pd.DataFrame({        "position": position, "spread": spread,        "ret_gross": ret_gross, "ret_net": ret_net,        "equity": equity, "turnover": turnover,    })def compute_metrics(bt: pd.DataFrame, periods: int = 252) -> dict:    """Performance metrics consistent with memory file §6.3."""    r  = bt["ret_net"].dropna()    eq = bt["equity"].dropna()    n_years = len(r) / periods    total   = (1 + r).prod()    ann_ret = total ** (1 / max(n_years, 0.01)) - 1    ann_vol = r.std() * np.sqrt(periods)    sharpe  = ann_ret / ann_vol if ann_vol > 0 else 0.0    running_max = eq.cummax()    dd = (eq - running_max) / running_max    max_dd  = float(-dd.min())    calmar  = ann_ret / max_dd if max_dd > 0 else 0.0    hit     = float((r > 0).sum() / len(r)) if len(r) > 0 else 0.0    ann_to  = bt["turnover"].sum() / max(n_years, 0.01)    return {        "Ann. Return": ann_ret, "Ann. Vol": ann_vol, "Sharpe": sharpe,        "Max DD": max_dd, "Calmar": calmar, "Hit Rate": hit,        "Ann. Turnover": ann_to,    }print("Backtest engine ready (single-asset + spread variants).")

## ResultsRun backtests across all three sub-modules and compute per-instrument metrics.

In [ ]:
results_all = {}metrics_rows = []# ── A. FX Carry ──print("=== FX Carry Backtests ===")for pair in fx_signals.columns:    if pair not in fx_data:        continue    spot = fx_data[pair]["spot"]    sig  = fx_signals[pair]    bt = backtest_single_asset(        spot, sig,        vol_target=met_cfg["vol_target_annual"],        vol_lookback=gcfg["vol_lookback_days"],        vol_cap=gcfg["vol_cap_multiplier"],    )    key = f"FX_{pair.upper()}"    results_all[key] = bt    m = compute_metrics(bt)    m["Instrument"] = key    m["Module"] = "FX Carry"    metrics_rows.append(m)    print(f"  {key:20s} Sharpe={m['Sharpe']:.2f}  MaxDD={m['Max DD']:.1%}")# ── B. Metals Calendar Spreads ──print("\n=== Metals Spread Backtests ===")spread_pairs = {    "GC_1v2": ("gc_fut_front", "gc_fut_second"),    "GC_2v3": ("gc_fut_second", "gc_fut_third"),    "SI_1v2": ("si_fut_front", "si_fut_second"),    "SI_2v3": ("si_fut_second", "si_fut_third"),}for label, (front, back) in spread_pairs.items():    if label not in metals_signals.columns:        continue    if front not in metals_prices.columns or back not in metals_prices.columns:        continue    spread = metals_prices[front] - metals_prices[back]    sig = metals_signals[label]    bt = backtest_spread(        spread, sig,        vol_target=met_cfg["vol_target_annual"],        vol_lookback=gcfg["vol_lookback_days"],    )    results_all[label] = bt    m = compute_metrics(bt)    m["Instrument"] = label    m["Module"] = "Metals Spread"    metrics_rows.append(m)    print(f"  {label:20s} Sharpe={m['Sharpe']:.2f}  MaxDD={m['Max DD']:.1%}")# ── C. SOFR Curve ──print("\n=== SOFR Curve Backtests ===")for label in sofr_signals.columns:    # Use front contract as proxy for PnL    if "fly" in label:        # Butterfly uses spread PnL        contracts = [c for c in sofr_cfg["instruments"] if c in sofr_prices.columns]        if len(contracts) >= 3:            fly_spread = sofr_prices[contracts[0]] - 2 * sofr_prices[contracts[1]] + sofr_prices[contracts[2]]            bt = backtest_spread(fly_spread, sofr_signals[label])        else:            continue    else:        # Adjacent slope trade        idx = int(label.split("_")[1].split("v")[0]) - 1        contracts = [c for c in sofr_cfg["instruments"] if c in sofr_prices.columns]        if idx + 1 < len(contracts):            slope = sofr_prices[contracts[idx + 1]] - sofr_prices[contracts[idx]]            bt = backtest_spread(slope, sofr_signals[label])        else:            continue    results_all[label] = bt    m = compute_metrics(bt)    m["Instrument"] = label    m["Module"] = "SOFR Curve"    metrics_rows.append(m)    print(f"  {label:20s} Sharpe={m['Sharpe']:.2f}  MaxDD={m['Max DD']:.1%}")# Summary tablemetrics_df = pd.DataFrame(metrics_rows).set_index("Instrument")fmt_df = metrics_df.drop(columns=["Module"]).copy()for col in ["Ann. Return", "Ann. Vol", "Max DD", "Hit Rate"]:    fmt_df[col] = fmt_df[col].map("{:.1%}".format)fmt_df["Sharpe"]  = fmt_df["Sharpe"].map("{:.2f}".format)fmt_df["Calmar"]  = fmt_df["Calmar"].map("{:.2f}".format)fmt_df["Ann. Turnover"] = fmt_df["Ann. Turnover"].map("{:.1f}x".format)print("\n" + "=" * 70)print("  Per-instrument metrics (IS period, 2 bp TC)")print("=" * 70)fmt_df

## Visualisations1. **Equity curves** — all sub-modules overlaid2. **FX carry ranking heatmap** — cross-sectional carry over time3. **Metals spread z-score map** — entry/exit levels4. **SOFR curve slope** — steepener/flattener regimes

In [ ]:
# --- 1. Equity Curves (all sub-modules) ---equity_df = pd.DataFrame({k: v["equity"] for k, v in results_all.items()})equity_norm = equity_df / equity_df.iloc[0]# Colour by modulemodule_colors = {    "FX Carry": "#3498db",    "Metals Spread": "#f39c12",    "SOFR Curve": "#2ecc71",}fig_eq = go.Figure()for inst in metrics_df.index:    if inst not in equity_norm.columns:        continue    module = metrics_df.loc[inst, "Module"]    fig_eq.add_trace(go.Scatter(        x=equity_norm.index,        y=equity_norm[inst],        name=inst,        line=dict(color=module_colors.get(module, "#95a5a6"), width=1.5),        legendgroup=module,        legendgrouptitle_text=module,    ))fig_eq.update_layout(    title="Carry Strategy — Equity Curves (IS: 2015-2022, $1M start)",    template="plotly_white",    hovermode="x unified",    legend=dict(orientation="h", y=-0.2),    height=550,    yaxis_title="Growth of $1",    yaxis_tickformat="$.2f",)fig_eq.show()# --- 2. FX Carry Ranking Heatmap ---# Compute annualised carry for each paircarry_matrix = pd.DataFrame()for pair, df in fx_data.items():    carry_ann = (-df["fwd_points"] / df["spot"]) * (365 / fx_cfg["carry_horizon_days"])    carry_matrix[pair.upper()] = carry_ann# Resample to monthly for readabilitycarry_monthly = carry_matrix.resample("ME").last().dropna()fig_carry = px.imshow(    (carry_monthly * 10_000).T.round(0),  # convert to bp    title="FX Implied Carry (bp annualised) — Monthly",    labels={"x": "", "y": "Pair", "color": "Carry (bp)"},    color_continuous_scale="RdYlGn",    aspect="auto",)fig_carry.update_layout(    template="plotly_white",    height=350,    xaxis=dict(dtick="M6", tickformat="%Y-%m"),)fig_carry.show()

## Metals Spread Z-Scores & SOFR Slope

In [ ]:
# --- 3. Metals Calendar Spread Z-Score Map ---if not metals_zscores.empty:    zs_monthly = metals_zscores.resample("ME").last().dropna()    fig_zs = px.imshow(        zs_monthly.T.round(2),        title="Metals Calendar Spread Z-Scores — Monthly",        labels={"x": "", "y": "Spread", "color": "Z-Score"},        color_continuous_scale="RdBu_r",        zmin=-3, zmax=3,        aspect="auto",    )    # Overlay entry/exit bands as annotations    fig_zs.update_layout(        template="plotly_white",        height=350,        xaxis=dict(dtick="M6", tickformat="%Y-%m"),    )    fig_zs.add_annotation(        text=f"Entry: ±{met_cfg['entry_zscore']}σ  |  Exit: ±{met_cfg['exit_zscore']}σ",        xref="paper", yref="paper", x=0.01, y=1.08,        showarrow=False, font=dict(size=11, color="gray"),    )    fig_zs.show()else:    print("No metals z-score data to plot.")# --- 4. SOFR Curve Slope ---if not sofr_slopes.empty:    fig_sofr = make_subplots(        rows=2, cols=1,        shared_xaxes=True,        row_heights=[0.6, 0.4],        subplot_titles=["SOFR Curve Slopes (price space)", "Slope Signals"],        vertical_spacing=0.10,    )    for col in sofr_slopes.columns:        fig_sofr.add_trace(            go.Scatter(                x=sofr_slopes.index, y=sofr_slopes[col],                name=col, mode="lines",                line=dict(width=1.5),            ),            row=1, col=1,        )    for col in sofr_signals.columns:        fig_sofr.add_trace(            go.Scatter(                x=sofr_signals.index, y=sofr_signals[col],                name=f"{col} sig", mode="lines",                line=dict(width=1, dash="dot"),            ),            row=2, col=1,        )    fig_sofr.update_layout(        title="SOFR Curve — Slopes & Signals (IS)",        template="plotly_white",        height=600,        legend=dict(orientation="h", y=-0.15),        hovermode="x unified",    )    fig_sofr.update_yaxes(title_text="Slope (ticks)", row=1, col=1)    fig_sofr.update_yaxes(title_text="Signal", range=[-1.2, 1.2], row=2, col=1)    fig_sofr.show()else:    print("No SOFR slope data to plot.")

## Performance AttributionBreak down PnL into gross carry contribution vs trend filter contributionper sub-module.

In [ ]:
# Aggregate returns by modulemodule_returns = {}for inst, row in metrics_df.iterrows():    module = row["Module"]    if module not in module_returns:        module_returns[module] = []    if inst in results_all:        module_returns[module].append(results_all[inst]["ret_net"])module_metrics = []for module, ret_list in module_returns.items():    combined = pd.concat(ret_list, axis=1).mean(axis=1)    eq = 1_000_000 * (1 + combined).cumprod()    m = compute_metrics(        pd.DataFrame({"ret_net": combined, "equity": eq, "turnover": pd.Series(0.0, index=combined.index)})    )    m["Module"] = module    module_metrics.append(m)mod_df = pd.DataFrame(module_metrics).set_index("Module")# Gross vs Net comparisongross_by_mod = {}net_by_mod = {}for inst, row in metrics_df.iterrows():    module = row["Module"]    if inst in results_all:        bt = results_all[inst]        if module not in gross_by_mod:            gross_by_mod[module] = []            net_by_mod[module] = []        gross_by_mod[module].append(bt["ret_gross"].sum())        net_by_mod[module].append(bt["ret_net"].sum())attrib = pd.DataFrame({    "Gross PnL (cum)": {m: np.mean(v) for m, v in gross_by_mod.items()},    "Net PnL (cum)":   {m: np.mean(v) for m, v in net_by_mod.items()},    "TC Drag (cum)":   {m: np.mean(gross_by_mod[m]) - np.mean(net_by_mod[m]) for m in gross_by_mod},})attrib.index.name = "Module"# Module-level metricsprint("=" * 65)print("  CARRY STRATEGY — MODULE-LEVEL ATTRIBUTION (IS: 2015-2022)")print("=" * 65)print()fmt_mod = mod_df.copy()for col in ["Ann. Return", "Ann. Vol", "Max DD", "Hit Rate"]:    fmt_mod[col] = fmt_mod[col].map("{:.1%}".format)fmt_mod["Sharpe"] = fmt_mod["Sharpe"].map("{:.2f}".format)fmt_mod["Calmar"] = fmt_mod["Calmar"].map("{:.2f}".format)fmt_mod["Ann. Turnover"] = fmt_mod["Ann. Turnover"].map("{:.1f}x".format)print("Module-level metrics:")display(fmt_mod)print()print("PnL attribution (average per instrument):")display(attrib.round(4))

## Performance SummaryCompare carry strategy average metrics against memory file targets (§6.3).Per-strategy Sharpe target is **> 0.5**.

In [ ]:
# Portfolio average across all carry instrumentsavg = metrics_df[["Ann. Return", "Ann. Vol", "Sharpe", "Max DD", "Calmar", "Hit Rate", "Ann. Turnover"]].mean()comparison = pd.DataFrame({    "Metric": [        "Sharpe Ratio",        "Annualised Vol",        "Max Drawdown",        "Calmar Ratio",        "Hit Rate",        "Ann. Turnover",    ],    "Target (§6.3)": [        f"> {targets['sharpe_per_strategy']:.1f}",        f"{targets['vol_range_annual'][0]:.0%} - {targets['vol_range_annual'][1]:.0%}",        f"< {targets['max_drawdown_pct']:.0f}%",        f"> {targets['calmar_ratio']:.1f}",        f"> {targets['hit_rate_daily']:.0%}",        f"< {targets['max_turnover_annual']:.0f}x",    ],    "Carry Average": [        f"{avg['Sharpe']:.2f}",        f"{avg['Ann. Vol']:.1%}",        f"{metrics_df['Max DD'].max():.1%}",        f"{avg['Calmar']:.2f}",        f"{avg['Hit Rate']:.1%}",        f"{avg['Ann. Turnover']:.1f}x",    ],}).set_index("Metric")print("=" * 65)print("  CARRY STRATEGY vs MEMORY FILE TARGETS  (IS: 2015-2022)")print("=" * 65)comparison

## ExportSave signals, equity curves, and summary tables to `outputs/`.

In [ ]:
output_dir = Path("../outputs")output_dir.mkdir(exist_ok=True)datestamp = datetime.now().strftime("%Y%m%d")# FX carry signalsfx_path = output_dir / f"fx_carry_signals_is_{datestamp}.csv"fx_signals.to_csv(fx_path)# Metals spread signalsspread_path = output_dir / f"metals_spread_signals_is_{datestamp}.csv"metals_signals.to_csv(spread_path)# SOFR signalssofr_path = output_dir / f"sofr_curve_signals_is_{datestamp}.csv"sofr_signals.to_csv(sofr_path)# Equity curveseq_path = output_dir / f"carry_equity_is_{datestamp}.csv"equity_df.to_csv(eq_path)# Summary HTMLhtml_path = output_dir / f"carry_summary_{datestamp}.html"html_content = (    "<h2>Carry Strategy — IS Performance (2015-2022)</h2>\n"    + "<h3>Per-instrument metrics</h3>\n"    + fmt_df.to_html()    + "<br><h3>Module-level attribution</h3>\n"    + fmt_mod.to_html()    + "<br><h3>vs Memory File Targets</h3>\n"    + comparison.to_html())with open(html_path, "w") as f:    f.write(html_content)print(f"Exported to {output_dir.resolve()}/")print(f"  {fx_path.name:45s}  ({fx_signals.shape[0]} rows)")print(f"  {spread_path.name:45s}  ({metals_signals.shape[0]} rows)")print(f"  {sofr_path.name:45s}  ({sofr_signals.shape[0]} rows)")print(f"  {eq_path.name}")print(f"  {html_path.name}")print(f"\nNotebook complete: {datetime.now():%Y-%m-%d %H:%M}")